# Capítulo 5 — Selección de modelos y comparación

En este capítulo aplicaremos distintos criterios de selección de modelos de regresión lineal múltiple para el sistema de bicicletas compartidas.

Trabajaremos con el conjunto de datos ya preprocesado `hour_prepared.csv` y:

- Definiremos la variable respuesta `cnt` y el conjunto de predictores.
- Ajustaremos un modelo completo (todas las variables numéricas).
- Implementaremos **selección hacia adelante (forward)** usando el **AIC**.
- Implementaremos **selección hacia atrás (backward)** usando el **AIC**.
- Compararemos el desempeño de los modelos seleccionados frente al modelo completo usando métricas sobre un conjunto de prueba.


In [2]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## 5.1 Carga del conjunto de datos y definición de variables

Cargamos el archivo preparado en capítulos anteriores y separamos:

- `y`: demanda de bicicletas por hora (`cnt`)
- `X_all`: todas las variables explicativas numéricas.


In [3]:
# Cargar el dataset preprocesado
df = pd.read_csv("../data/hour_prepared.csv")

# Variable respuesta
y = df["cnt"].astype(float)

# Predictores: quitamos la respuesta
X_all = df.drop(columns=["cnt"])

# Nos quedamos solo con columnas numéricas y forzamos a float
X_all = X_all.select_dtypes(include=["number"]).astype(float)

X_all.shape, X_all.dtypes.head()

((17379, 5),
 temp         float64
 atemp        float64
 hum          float64
 windspeed    float64
 peak_hour    float64
 dtype: object)

## 5.2 Partición entrenamiento / prueba y modelo completo

Dividimos el conjunto de datos en entrenamiento (80%) y prueba (20%) para poder comparar el desempeño fuera de la muestra.

Ajustamos primero un modelo **completo** con todas las variables numéricas para tener un punto de referencia.


In [4]:
# Partición de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=123
)

# Función auxiliar para ajustar OLS dado un subconjunto de columnas
def fit_ols_from_cols(cols, X_source, y_source):
    """
    Ajusta un modelo OLS usando las columnas indicadas en 'cols'
    de X_source, añadiendo un intercepto.
    """
    Xc = sm.add_constant(X_source[cols], has_constant="add")
    model = sm.OLS(y_source, Xc).fit()
    return model

# Modelo completo con todas las columnas numéricas
all_vars = list(X_all.columns)
ols_full = fit_ols_from_cols(all_vars, X_train, y_train)

ols_full.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    cnt   R-squared:                       0.452
Model:                            OLS   Adj. R-squared:                  0.452
Method:                 Least Squares   F-statistic:                     2293.
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        02:35:58   Log-Likelihood:                -87821.
No. Observations:               13903   AIC:                         1.757e+05
Df Residuals:                   13897   BIC:                         1.757e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        120.0733      6.250     19.212      0.000     107.823     132.324
temp          61.2861     40.274      1.522      0.128     -17.656     140.229
atemp        340.4978     45.201      7.533      0.000     251.899     429.097
hum         -273.4382      6.176    -44.276      0.000    -285.543    -261.333
windspeed      9.5407     10.030      0.951      0.341     -10.119      29.200
peak_hour    184.7146      2.622     70.437      0.000     179.574     189.855
==============================================================================
Omnibus:                     1672.276   Durbin-Watson:                   2.033
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             2797.170
Skew:                           0.831   Prob(JB):                         0.00
Kurtosis:                       4.438   Cond. No.                         75.1
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Calculamos el desempeño del modelo completo en el conjunto de prueba usando la raíz del error cuadrático medio (RMSE).

In [5]:
# Desempeño del modelo completo en prueba
X_test_const_full = sm.add_constant(X_test[all_vars], has_constant="add")
y_pred_full = ols_full.predict(X_test_const_full)

rmse_full = mean_squared_error(y_test, y_pred_full) ** 0.5
rmse_full

135.8479540053958

## 5.3 Selección hacia adelante (Forward AIC)

Partimos de un modelo sin predictores y vamos agregando, uno a uno, las variables que produzcan la **mayor reducción del AIC**, hasta que ninguna variable mejore el criterio.

Guardamos el historial de selección para analizar el orden en que entran las variables.


In [6]:
candidate_vars = list(X_all.columns)

selected_forward = []
current_aic = np.inf
history_forward = []

while True:
    remaining = [v for v in candidate_vars if v not in selected_forward]
    if not remaining:
        break

    scores = []
    for var in remaining:
        cols_trial = selected_forward + [var]
        model_trial = fit_ols_from_cols(cols_trial, X_train, y_train)
        scores.append((model_trial.aic, var, model_trial))

    # Ordenamos por AIC
    scores.sort(key=lambda x: x[0])
    best_aic, best_var, best_model = scores[0]

    # Aceptamos la nueva variable solo si mejora el AIC
    if best_aic < current_aic - 1e-6:
        current_aic = best_aic
        selected_forward.append(best_var)
        history_forward.append((len(selected_forward), best_var, best_aic))
    else:
        break

selected_forward, current_aic

(['peak_hour', 'temp', 'hum', 'atemp'], np.float64(175653.4250194364))

Mostramos el historial de entrada de variables según el criterio de AIC.

In [7]:
history_forward_df = pd.DataFrame(
    history_forward, columns=["k", "variable_agregada", "AIC"]
)
history_forward_df

,k,variable_agregada,AIC
0,1,peak_hour,180849.397135
1,2,temp,177643.260976
2,3,hum,175708.042095
3,4,atemp,175653.425019


Evaluamos el modelo resultante de selección forward en el conjunto de prueba.

In [8]:
# Ajustar modelo final por forward en todo X_train
ols_forward = fit_ols_from_cols(selected_forward, X_train, y_train)

# Predicción en prueba
X_test_const_forward = sm.add_constant(X_test[selected_forward], has_constant="add")
y_pred_forward = ols_forward.predict(X_test_const_forward)

rmse_forward = mean_squared_error(y_test, y_pred_forward) ** 0.5

rmse_full, rmse_forward

(135.8479540053958, 135.86837940388838)

## 5.4 Selección hacia atrás (Backward AIC)

Ahora comenzamos desde el modelo **completo** y vamos eliminando, una a una, las variables que menos aportan según el AIC, hasta que no haya mejoras.

Esta estrategia suele ser útil cuando partimos de un conjunto grande de predictores y queremos simplificar el modelo.


In [9]:
selected_backward = list(X_all.columns)
current_aic_b = fit_ols_from_cols(selected_backward, X_train, y_train).aic
history_backward = []

while True:
    scores = []
    # probamos eliminar de a una variable
    if len(selected_backward) == 1:
        break

    for var in selected_backward:
        cols_trial = [v for v in selected_backward if v != var]
        model_trial = fit_ols_from_cols(cols_trial, X_train, y_train)
        scores.append((model_trial.aic, var, cols_trial, model_trial))

    scores.sort(key=lambda x: x[0])
    best_aic, removed_var, best_cols, best_model = scores[0]

    if best_aic < current_aic_b - 1e-6:
        current_aic_b = best_aic
        selected_backward = best_cols
        history_backward.append(
            (len(selected_backward), removed_var, best_aic)
        )
    else:
        break

selected_backward, current_aic_b

(['temp', 'atemp', 'hum', 'peak_hour'], np.float64(175653.4250194364))

Mostramos cómo fue quedando el modelo en el proceso de eliminación hacia atrás.

In [10]:
history_backward_df = pd.DataFrame(
    history_backward, columns=["k", "variable_eliminada", "AIC"]
)
history_backward_df

,k,variable_eliminada,AIC
0,4,windspeed,175653.425019


Evaluamos el modelo resultante de selección backward en el conjunto de prueba y lo comparamos con el modelo completo y el modelo forward.

In [11]:
ols_backward = fit_ols_from_cols(selected_backward, X_train, y_train)
X_test_const_backward = sm.add_constant(X_test[selected_backward], has_constant="add")
y_pred_backward = ols_backward.predict(X_test_const_backward)

rmse_backward = mean_squared_error(y_test, y_pred_backward) ** 0.5

rmse_full, rmse_forward, rmse_backward

(135.8479540053958, 135.86837940388838, 135.8683794038884)

## 5.5 Resumen comparativo de modelos

Construimos una tabla de comparación de los modelos considerados:

- Modelo completo (todas las variables).
- Modelo forward (AIC).
- Modelo backward (AIC).

Usamos como referencia el RMSE en el conjunto de prueba.


In [12]:
resumen_modelos = pd.DataFrame({
    "modelo": ["Completo", "Forward AIC", "Backward AIC"],
    "k_variables": [len(all_vars), len(selected_forward), len(selected_backward)],
    "RMSE_test": [rmse_full, rmse_forward, rmse_backward],
})

resumen_modelos

,modelo,k_variables,RMSE_test
0,Completo,5,135.847954
1,Forward AIC,4,135.868379
2,Backward AIC,4,135.868379


En capítulos posteriores, tomaremos como punto de partida el modelo que muestre mejor compromiso entre complejidad (número de variables) y error de predicción en el conjunto de prueba.

También podremos usar el conjunto de variables seleccionadas como base para ajustes más avanzados (regularización, validación cruzada más exhaustiva, etc.).